# Training artifacts — EDA & model inspection

Scans `training/**/artifacts` for **Parquet**, **PKL**, and **CatBoost .cbm** files.  
Run the repo from the project root, or from `training/`, and adjust `REPO_ROOT` in the next cell if paths don’t resolve.

**Optional deps** (for SHAP): `pip install shap matplotlib`  
**CatBoost** required for `.cbm`: `pip install catboost`  

**Pickled sklearn pipelines** (`preprocessor*.pkl`): unpickling can fail if `scikit-learn` differs from the version used when saving. Install **`scikit-learn==1.5.2`** (see `training/requirements.txt`), restart the kernel, and rerun.

In [ ]:
from __future__ import annotations

%matplotlib inline

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# Resolve repo root (run notebook from repo root or from training/)
_here = Path.cwd().resolve()
if (_here / "training" / "v2" / "artifacts").is_dir():
    REPO_ROOT = _here
elif (_here / "v2" / "artifacts").is_dir():
    REPO_ROOT = _here.parent
else:
    REPO_ROOT = _here

TRAINING = REPO_ROOT / "training"
ARTIFACT_GLOBS = ["**/artifacts/*.parquet", "**/artifacts/*.pkl", "**/artifacts/*.cbm", "**/artifacts/*.pt"]

def list_artifacts():
    files = []
    for pattern in ARTIFACT_GLOBS:
        files.extend(sorted(TRAINING.glob(pattern)))
    return sorted(set(files))

try:
    import sklearn

    print("scikit-learn:", sklearn.__version__, "(pin 1.5.2 in training/requirements.txt for pickled preprocessors)")
except ImportError:
    print("scikit-learn: not installed")

files = list_artifacts()
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"Found {len(files)} artifact files under training/**/artifacts")
for p in files:
    print(" -", p.relative_to(TRAINING))

## Parquet files
Shape, dtypes, `info()`, head, and numeric summaries.

In [ ]:
def eda_parquet(path: Path) -> None:
    print("=" * 72)
    print(f"PARQUET: {path.relative_to(TRAINING)}")
    print("=" * 72)
    df = pd.read_parquet(path)
    print("shape:", df.shape)
    display(df.head(8))
    df.info()
    num = df.select_dtypes(include=[np.number])
    if len(num.columns):
        display(num.describe().T)
    obj = df.select_dtypes(include=["object", "string"])
    if len(obj.columns):
        print("--- sample categorical value counts (top 5 each) ---")
        for c in obj.columns[:12]:
            print(c, obj[c].value_counts().head(5).to_dict())
        if len(obj.columns) > 12:
            print(f"... ({len(obj.columns) - 12} more object columns)")


parquets = [p for p in files if p.suffix == ".parquet"]
for p in parquets:
    eda_parquet(p)

## Pickle (`.pkl`) files
Load with `joblib`, print type and a short structural summary (dict keys, DataFrame shape, sklearn estimator, etc.).

In [ ]:
def summarize_obj(name: str, obj, depth: int = 0) -> None:
    ind = "  " * depth
    if isinstance(obj, dict):
        print(f"{ind}{name}: dict, len={len(obj)}")
        for k in list(obj.keys())[:20]:
            summarize_obj(str(k), obj[k], depth + 1)
        if len(obj) > 20:
            print(f"{ind}  ... ({len(obj) - 20} more keys)")
    elif isinstance(obj, (list, tuple)):
        print(f"{ind}{name}: {type(obj).__name__}, len={len(obj)}")
        if len(obj) and depth < 2:
            summarize_obj("[0]", obj[0], depth + 1)
    elif isinstance(obj, pd.DataFrame):
        print(f"{ind}{name}: DataFrame {obj.shape}")
        display(obj.head(3))
    elif hasattr(obj, "get_params"):
        try:
            print(f"{ind}{name}: {type(obj).__name__}")
            print(f"{ind}  params: {obj.get_params()}")
        except Exception as e:
            print(f"{ind}{name}: {type(obj).__name__} (get_params failed: {e})")
    else:
        s = str(type(obj).__name__)
        extra = ""
        if hasattr(obj, "shape"):
            extra = f" shape={getattr(obj, 'shape', None)}"
        print(f"{ind}{name}: {s}{extra}")


def eda_pickle(path: Path) -> None:
    import pickle as _pickle

    print("=" * 72)
    print(f"PKL: {path.relative_to(TRAINING)}")
    print("=" * 72)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        try:
            obj = joblib.load(path)
        except (AttributeError, ModuleNotFoundError, _pickle.UnpicklingError, EOFError) as e:
            try:
                import sklearn

                skv = sklearn.__version__
            except Exception:
                skv = "?"
            print(
                f"Could not unpickle ({type(e).__name__}): {e}\n"
                "Usually the pickle was built with another scikit-learn (see ColumnTransformer internals).\n"
                f"This kernel's scikit-learn: {skv}\n"
                "Try aligning with training/requirements.txt:\n"
                "  pip install 'scikit-learn==1.5.2'\n"
                "Then restart the Jupyter kernel and rerun.\n"
                "Or rebuild preprocessor_*.pkl with your current sklearn."
            )
            return
    summarize_obj("root", obj)


pkls = [p for p in files if p.suffix == ".pkl"]
for p in pkls:
    eda_pickle(p)

## CatBoost `.cbm`
- Load model and print basic metadata.
- **Feature importance** (built-in).
- **SHAP** (optional): uses `metrics_*.json` next to the `.cbm` when present to rebuild column layout; background rows are **random placeholders** for visualization only — interpret SHAP as qualitative, not causal.

If no sibling metrics JSON is found, importance still runs using `feature_names` from the model when available.

In [ ]:
def find_metrics_json(cbm_path: Path) -> Path | None:
    """Pick metrics JSON that lists model features (skip legacy metrics.json without 'features')."""
    d = cbm_path.parent
    stem = cbm_path.stem  # e.g. model_catboost_v2
    paired: list[Path] = []
    if stem.startswith("model_"):
        paired.append(d / f"metrics_{stem[len('model_'):]}.json")
    paired.extend(sorted(d.glob("metrics*.json")))

    seen: set[Path] = set()
    best_path: Path | None = None
    best_n = -1
    for p in paired:
        p = p.resolve()
        if p in seen or not p.is_file():
            continue
        seen.add(p)
        try:
            with open(p, encoding="utf-8") as f:
                m = json.load(f)
            feats = m.get("features")
            n = len(feats) if isinstance(feats, list) else 0
            if n > best_n:
                best_n = n
                best_path = p
        except Exception:
            continue
    return best_path


def synthetic_pool_from_metrics(metrics_path: Path, n: int = 400):
    from catboost import Pool

    with open(metrics_path, encoding="utf-8") as f:
        m = json.load(f)
    feat = m.get("features") or []
    if not feat:
        raise ValueError(f"{metrics_path.name} has no non-empty 'features' list — cannot build Pool")
    cat_names = set(m.get("cat_features") or [])
    rng = np.random.default_rng(42)
    data = {}
    for name in feat:
        if name in cat_names:
            data[name] = rng.choice(["A", "B", "C", "D", "E"], size=n)
        else:
            data[name] = rng.normal(0, 1, size=n) * 50
    dfX = pd.DataFrame(data)
    cat_idx = [i for i, c in enumerate(feat) if c in cat_names]
    pool = Pool(dfX, cat_features=cat_idx)
    return pool, dfX, feat, m


def analyze_catboost_cbm(cbm_path: Path) -> None:
    import matplotlib.pyplot as plt
    from catboost import CatBoostRegressor

    print("=" * 72)
    print(f"CBM: {cbm_path.relative_to(TRAINING)}")
    print("=" * 72)
    model = CatBoostRegressor()
    model.load_model(str(cbm_path))
    print("tree_count:", model.tree_count_)
    fn = model.feature_names_
    if fn is None:
        print("feature_names_: (none stored in file)")
    else:
        print("n_features:", len(fn))

    metrics_path = find_metrics_json(cbm_path)
    names_for_importance = list(fn) if fn is not None else None
    if names_for_importance is None and metrics_path:
        with open(metrics_path, encoding="utf-8") as f:
            names_for_importance = json.load(f).get("features")

    fi = model.get_feature_importance()
    if names_for_importance and len(names_for_importance) == len(fi):
        fi_s = pd.Series(fi, index=names_for_importance).sort_values(ascending=True)
    else:
        fi_s = pd.Series(fi, index=[f"f{i}" for i in range(len(fi))]).sort_values(ascending=True)
        print("Warning: feature name count mismatch; using generic labels.")

    ax = fi_s.tail(25).plot.barh(figsize=(8, 10), title="CatBoost feature importance (top 25)")
    ax.set_xlabel("Importance")
    display(fi_s.tail(25).to_frame("importance"))
    plt.tight_layout()
    plt.show()

    # SHAP (optional)
    try:
        import shap
    except ImportError as e:
        print("SHAP not installed — skip SHAP.", e)
        return

    if not metrics_path:
        print("No usable metrics*.json beside .cbm (need JSON with 'features' array) — skip SHAP.")
        return

    try:
        pool, dfX, feat_list, _ = synthetic_pool_from_metrics(metrics_path)
    except ValueError as e:
        print("SHAP skipped:", e)
        return
    if len(fi) != len(feat_list):
        print("SHAP skipped: metrics feature count != model importance length")
        return

    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(pool)
    except Exception as e:
        print("SHAP TreeExplainer failed:", e)
        return

    print("SHAP values shape:", np.asarray(shap_values).shape)
    shap.summary_plot(shap_values, dfX, max_display=20, show=False)
    plt.tight_layout()
    plt.show()
    print("Note: SHAP used synthetic random rows — use real validation rows for rigorous interpretation.")


cbms = [p for p in files if p.suffix == ".cbm"]
for p in cbms:
    analyze_catboost_cbm(p)

## PyTorch `.pt` (if present)
Lightweight `torch.load` / state_dict key listing — no full forward pass.

In [ ]:
def eda_torch_pt(path: Path) -> None:
    print("=" * 72)
    print(f"PT: {path.relative_to(TRAINING)}")
    print("=" * 72)
    try:
        import torch
    except ImportError:
        print("torch not installed")
        return
    try:
        w = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        w = torch.load(path, map_location="cpu")
    print("Loaded type:", type(w))
    if isinstance(w, dict):
        print("Top-level keys:", list(w.keys())[:40])
        for k, v in list(w.items())[:5]:
            if hasattr(v, "shape"):
                print(f"  {k}: tensor/array shape {tuple(v.shape)}")
            else:
                print(f"  {k}: {type(v).__name__}")
    summarize_obj("checkpoint", w)


pts = [p for p in files if p.suffix == ".pt"]
for p in pts:
    eda_torch_pt(p)

## JSON metrics (same folders)
Pretty-print small metrics files for quick comparison.

In [ ]:
json_files = sorted(TRAINING.glob("**/artifacts/*.json"))
for jp in json_files:
    print("=" * 72)
    print(jp.relative_to(TRAINING))
    with open(jp, encoding="utf-8") as f:
        data = json.load(f)
    print(json.dumps(data, indent=2)[:8000])
    if len(json.dumps(data)) > 8000:
        print("\n... (truncated)")